# AWS EC2 Instance Pricing Prediction - Complete ML Pipeline

This notebook implements a comprehensive machine learning pipeline to:
- **Find model suitable with the dataset**
- **Identify matching parameters**
- **Build modelling algorithm**
- **Optimize model parameters**
- **Model testing**
- **Evaluate results**

**Dataset**: AWS EC2 Instance Comparison - predicting hourly instance costs based on specifications

## Section 1: Import Libraries & Load Data

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ML Libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score, cross_validate, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Advanced models
import xgboost as xgb
import lightgbm as lgb

# SHAP for interpretability
import shap

# Utilities
import joblib
from scipy.stats import randint, uniform
import optuna
from optuna.samplers import TPESampler

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

# Load dataset
data_path = Path('c:/Users/15132000230/Documents/GitHub/sintance-DataScientistSPV-KOMDIGI/data/instance_comparison.csv')
df = pd.read_csv(data_path)

print(f"\n✓ Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)
print(f"\nMemory usage:")
print(df.memory_usage(deep=True))

## Section 2: Inspect Dataset & Detect Problem Type

In [ ]:
# Extract target variable: Linux On Demand Cost
# First, let's examine the cost columns
cost_columns = [col for col in df.columns if 'On Demand cost' in col]
print("Cost columns available:")
for col in cost_columns:
    print(f"  - {col}")

# We'll use Linux On Demand cost as our target
target_column = 'Linux On Demand cost'
print(f"\n✓ Target variable selected: {target_column}")

# Convert price strings to numeric
def extract_numeric(price_str):
    """Extract numeric value from price string like '$0.272 hourly'"""
    if pd.isna(price_str) or price_str == 'unavailable':
        return np.nan
    try:
        return float(price_str.replace('$', '').replace(' hourly', '').strip())
    except:
        return np.nan

# Create target column
df['target_cost'] = df[target_column].apply(extract_numeric)

print(f"\nTarget variable statistics:")
print(df['target_cost'].describe())

# Check for missing values in target
print(f"\nMissing values in target: {df['target_cost'].isna().sum()}")
print(f"Problem type: REGRESSION (predicting continuous hourly costs)")

# Display distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
df['target_cost'].dropna().hist(bins=30, edgecolor='black')
plt.xlabel('Hourly Cost ($)')
plt.ylabel('Frequency')
plt.title('Distribution of Linux On Demand Cost')
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
df['target_cost'].dropna().plot(kind='box', vert=True)
plt.ylabel('Hourly Cost ($)')
plt.title('Boxplot of Linux On Demand Cost')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Dataset inspected and problem type identified as REGRESSION")

## Section 3: Quick Data Quality Checks

In [ ]:
# Data quality checks
print("="*60)
print("DATA QUALITY ASSESSMENT")
print("="*60)

# 1. Missing values check
print("\n1. MISSING VALUES:")
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_counts, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(f"Columns with missing values: {len(missing_df)}")
if len(missing_df) > 0:
    print(missing_df.head(10))

# 2. Duplicates
print(f"\n2. DUPLICATES:")
print(f"Duplicate rows: {df.duplicated().sum()}")

# 3. Extract numeric columns (excluding target)
def safe_numeric_convert(val):
    """Try to extract numeric from various formats"""
    if pd.isna(val) or val == 'unknown' or val == 'unavailable':
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    try:
        # Handle strings with spaces and special units
        if isinstance(val, str):
            # Remove units like GHz, GiB, Gbps, IOPS, etc.
            val_clean = val.split()[0]  # Take first part before units
            val_clean = val_clean.replace(',', '')  # Remove commas
            return float(val_clean)
        return np.nan
    except:
        return np.nan

# Convert relevant columns to numeric
print(f"\n3. FEATURE TYPE DETECTION:")
numeric_candidates = ['Memory', 'vCPUs', 'ECU per vCPU', 'Clock Speed(GHz)', 
                      'Network Performance', 'EBS Optimized: Max Bandwidth',
                      'EBS Optimized: Max Throughput (128K)', 'EBS Optimized: Max IOPS (16K)',
                      'Max IPs', 'Max ENIs']

for col in numeric_candidates:
    if col in df.columns:
        df[col] = df[col].apply(safe_numeric_convert)

print(f"Total columns: {len(df.columns)}")
print(f"Numeric columns identified: {df.select_dtypes(include=[np.number]).shape[1]}")
print(f"Object (categorical) columns: {df.select_dtypes(include='object').shape[1]}")

# 4. Summary statistics
print(f"\n4. SAMPLES WITH COMPLETE TARGET:")
df_with_target = df[df['target_cost'].notna()].copy()
print(f"Total samples: {len(df)}")
print(f"Samples with target cost: {len(df_with_target)}")
print(f"Data completeness: {(len(df_with_target)/len(df))*100:.1f}%")

# Remove samples without target for modeling
df = df_with_target.copy()
print(f"\n✓ Using {len(df)} samples with complete target values")

# 5. Check for infinite values
print(f"\n5. INFINITE VALUES CHECK:")
for col in df.select_dtypes(include=[np.number]).columns:
    inf_count = np.isinf(df[col]).sum()
    if inf_count > 0:
        print(f"  {col}: {inf_count} infinite values")

print("\n✓ Data quality checks completed!")

## Section 4: Exploratory Data Analysis (EDA)

In [ ]:
# Get numeric features for EDA
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('target_cost')  # Remove target from features

print(f"Numeric features for modeling: {len(numeric_cols)}")
print(f"Features: {numeric_cols[:10]}... (showing first 10)")

# Select top features for visualization (those with less missing values)
missing_count = df[numeric_cols].isnull().sum()
top_features = missing_count[missing_count < len(df) * 0.2].index.tolist()[:6]
print(f"\nTop features for visualization: {top_features}")

# 1. Distribution plots for top numeric features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, col in enumerate(top_features):
    if col != 'target_cost':
        data_to_plot = df[col].dropna()
        if len(data_to_plot) > 0:
            axes[idx].hist(data_to_plot, bins=30, edgecolor='black', alpha=0.7)
            axes[idx].set_title(f'Distribution of {col}')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Frequency')
            axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# 2. Correlation with target
print("\n" + "="*60)
print("CORRELATION WITH TARGET (Top 10)")
print("="*60)
correlations = {}
for col in numeric_cols:
    # Calculate correlation only for pairs with sufficient data
    valid_data = df[[col, 'target_cost']].dropna()
    if len(valid_data) > 10:
        corr = valid_data[col].corr(valid_data['target_cost'])
        correlations[col] = corr

# Sort by absolute correlation
correlations_sorted = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
print("\nFeature - Correlation with Target Cost:")
for feat, corr in correlations_sorted[:10]:
    print(f"  {feat:35s}: {corr:7.4f}")

# 3. Correlation heatmap (top features)
top_corr_features = [feat for feat, _ in correlations_sorted[:8]]
if len(top_corr_features) > 0:
    corr_data = df[top_corr_features + ['target_cost']].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Correlation Heatmap - Top Features vs Target Cost')
    plt.tight_layout()
    plt.show()

# 4. Scatter plots of top features vs target
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (feat, _) in enumerate(correlations_sorted[:6]):
    valid_data = df[[feat, 'target_cost']].dropna()
    axes[idx].scatter(valid_data[feat], valid_data['target_cost'], alpha=0.6, s=50)
    axes[idx].set_xlabel(feat)
    axes[idx].set_ylabel('Target Cost ($)')
    axes[idx].set_title(f'{feat} vs Target Cost')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ EDA completed!")

## Section 5: Preprocessing Pipeline

In [ ]:
# Prepare features and target
y = df['target_cost'].copy()

# Select features: numeric columns with acceptable missing rates
X = df[numeric_cols].copy()

# Keep only features with < 50% missing values
X = X.loc[:, X.isnull().mean() < 0.5]
feature_cols = X.columns.tolist()

print("="*60)
print("PREPROCESSING PIPELINE SETUP")
print("="*60)
print(f"\nFeatures selected: {len(feature_cols)}")
print(f"Target variable: target_cost")
print(f"Dataset shape: X={X.shape}, y={y.shape}")

# Handle outliers using IQR method
def clip_outliers(series, lower_percentile=1, upper_percentile=99):
    """Clip outliers based on percentiles"""
    q1 = series.quantile(lower_percentile/100)
    q3 = series.quantile(upper_percentile/100)
    return series.clip(lower=q1, upper=q3)

# For target variable, clip extreme outliers
y_clipped = clip_outliers(y, lower_percentile=1, upper_percentile=99)
outlier_removed = (y != y_clipped).sum()
print(f"\nOutliers removed from target: {outlier_removed}")

# Create preprocessing pipeline
print("\n" + "-"*60)
print("BUILDING SKLEARN PREPROCESSING PIPELINE")
print("-"*60)

# Numeric transformer: median imputation + robust scaling
numeric_transformer = Pipeline(steps=[
    ('imputer', pd.Series.fillna),  # Use forward fill for time series, or median
    ('scaler', RobustScaler())  # RobustScaler handles outliers better than StandardScaler
])

# For sklearn ColumnTransformer, we use SimpleImputer
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# Create ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, feature_cols)
    ],
    remainder='drop'
)

print("✓ Preprocessing pipeline created:")
print(f"  - Numeric Imputation: Median")
print(f"  - Scaling: RobustScaler (resistant to outliers)")

# Fit and transform data to verify
X_preprocessed = preprocessor.fit_transform(X)
print(f"\n✓ Preprocessing applied successfully!")
print(f"  - Original shape: {X.shape}")
print(f"  - Preprocessed shape: {X_preprocessed.shape}")
print(f"  - No missing values: {np.isnan(X_preprocessed).sum() == 0}")

# Keep a clean version for later use
X_clean = X.copy()
y_clean = y_clipped.copy()

## Section 6: Feature Engineering & Selection

In [ ]:
print("="*60)
print("FEATURE ENGINEERING & SELECTION")
print("="*60)

# Apply preprocessing to get clean data
X_preprocessed_clean = preprocessor.fit_transform(X_clean)
X_preprocessed_df = pd.DataFrame(X_preprocessed_clean, columns=feature_cols)

# 1. Variance Threshold - remove low variance features
print("\n1. VARIANCE THRESHOLD ANALYSIS:")
variance_threshold = VarianceThreshold(threshold=0.01)
variance_threshold.fit(X_preprocessed_df)
selected_mask = variance_threshold.get_support()
high_variance_features = X_preprocessed_df.columns[selected_mask].tolist()
print(f"   Features with variance > 0.01: {len(high_variance_features)}/{len(feature_cols)}")

# 2. SelectKBest - top features by regression score
print("\n2. SELECTKBEST (F-REGRESSION) - Top 15 Features:")
selectkbest = SelectKBest(score_func=f_regression, k=min(15, len(feature_cols)))
selectkbest.fit(X_preprocessed_df, y_clean)
top_k_features = X_preprocessed_df.columns[selectkbest.get_support()].tolist()

scores = selectkbest.scores_
feature_scores = pd.DataFrame({
    'Feature': feature_cols,
    'F-Score': scores
}).sort_values('F-Score', ascending=False)

print(feature_scores.head(15))

# 3. Tree-based feature importance using RandomForest
print("\n3. TREE-BASED FEATURE IMPORTANCE (Random Forest):")
rf_importance = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf_importance.fit(X_preprocessed_df, y_clean)

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_importance.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance_df.head(10))

# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 SelectKBest
axes[0].barh(feature_scores.head(10)['Feature'], feature_scores.head(10)['F-Score'])
axes[0].set_xlabel('F-Score')
axes[0].set_title('Top 10 Features by F-Regression Score')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Top 10 Tree Importance
axes[1].barh(importance_df.head(10)['Feature'], importance_df.head(10)['Importance'])
axes[1].set_xlabel('Importance')
axes[1].set_title('Top 10 Features by Random Forest Importance')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Select final features: combination of all methods
final_features = list(set(high_variance_features) & set(top_k_features) & set(importance_df.head(15)['Feature'].tolist()))
if len(final_features) < 10:
    # If intersection is too small, use top features from importance
    final_features = importance_df.head(15)['Feature'].tolist()

final_features = sorted(final_features)
print(f"\n✓ Final selected features ({len(final_features)}): ")
for i, feat in enumerate(final_features, 1):
    print(f"  {i:2d}. {feat}")

# Store for later use
X_selected = X_preprocessed_df[final_features].copy()

## Section 7: Train/Validation/Test Split & CV Strategy

In [ ]:
# Train/Validation/Test Split: 60/20/20
random_state = 42

print("="*60)
print("TRAIN/VALIDATION/TEST SPLIT")
print("="*60)

# First split: 80% train+val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X_selected, y_clean, test_size=0.2, random_state=random_state
)

# Second split: 75% train, 25% val (from temp)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=random_state
)

print(f"\nTrain set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X_selected)*100:.1f}%)")
print(f"Val set:   {X_val.shape[0]} samples ({X_val.shape[0]/len(X_selected)*100:.1f}%)")
print(f"Test set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X_selected)*100:.1f}%)")

print(f"\nTarget distribution:")
print(f"  Train: mean=${y_train.mean():.2f}, std=${y_train.std():.2f}")
print(f"  Val:   mean=${y_val.mean():.2f}, std=${y_val.std():.2f}")
print(f"  Test:  mean=${y_test.mean():.2f}, std=${y_test.std():.2f}")

# Cross-validation strategy for regression
print(f"\n" + "-"*60)
print("CROSS-VALIDATION STRATEGY")
print("-"*60)

cv = KFold(n_splits=5, shuffle=True, random_state=random_state)
print(f"✓ Using 5-Fold Cross-Validation for Regression")
print(f"  - Ensures balanced evaluation across all data")
print(f"  - Reduces variance in performance estimates")
print(f"  - Better generalization assessment")

# Store splits info
splits_info = {
    'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
    'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
    'cv': cv, 'random_state': random_state, 'final_features': final_features
}

print(f"\n✓ Train/Val/Test split completed and stored!")
print(f"✓ Random state: {random_state} (for reproducibility)")

## Section 8: Baseline Models

In [ ]:
print("="*60)
print("BASELINE MODELS - SIMPLE BENCHMARKS")
print("="*60)

# Metric function
def evaluate_regression(y_true, y_pred, model_name="Model"):
    """Calculate regression metrics"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'MSE': mse}

baseline_results = []

# Baseline 1: Mean predictor (predicts mean of training set)
print("\n1. MEAN PREDICTOR (Baseline):")
mean_pred_train = np.full_like(y_train, y_train.mean(), dtype=float)
mean_pred_val = np.full_like(y_val, y_train.mean(), dtype=float)
mean_pred_test = np.full_like(y_test, y_train.mean(), dtype=float)

result = evaluate_regression(y_val, mean_pred_val, "Mean Predictor")
baseline_results.append(result)
print(f"   Train RMSE: ${np.sqrt(mean_squared_error(y_train, mean_pred_train)):.4f}")
print(f"   Val RMSE:   ${result['RMSE']:.4f}, MAE: ${result['MAE']:.4f}, R²: {result['R²']:.4f}")

# Baseline 2: Linear Regression (simple)
print("\n2. LINEAR REGRESSION:")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_train_pred = lr_model.predict(X_train)
lr_val_pred = lr_model.predict(X_val)
lr_test_pred = lr_model.predict(X_test)

result = evaluate_regression(y_val, lr_val_pred, "Linear Regression")
baseline_results.append(result)
print(f"   Train RMSE: ${np.sqrt(mean_squared_error(y_train, lr_train_pred)):.4f}")
print(f"   Val RMSE:   ${result['RMSE']:.4f}, MAE: ${result['MAE']:.4f}, R²: {result['R²']:.4f}")

# Baseline 3: Ridge Regression
print("\n3. RIDGE REGRESSION:")
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)

ridge_train_pred = ridge_model.predict(X_train)
ridge_val_pred = ridge_model.predict(X_val)
ridge_test_pred = ridge_model.predict(X_test)

result = evaluate_regression(y_val, ridge_val_pred, "Ridge Regression")
baseline_results.append(result)
print(f"   Train RMSE: ${np.sqrt(mean_squared_error(y_train, ridge_train_pred)):.4f}")
print(f"   Val RMSE:   ${result['RMSE']:.4f}, MAE: ${result['MAE']:.4f}, R²: {result['R²']:.4f}")

# Cross-validation results
print("\n" + "-"*60)
print("CROSS-VALIDATION RESULTS (5-Fold):")
print("-"*60)

cv_results = {
    'Linear Regression': cross_val_score(lr_model, X_train, y_train, cv=cv, 
                                         scoring='r2', n_jobs=-1),
    'Ridge Regression': cross_val_score(ridge_model, X_train, y_train, cv=cv, 
                                        scoring='r2', n_jobs=-1)
}

for model_name, scores in cv_results.items():
    print(f"{model_name}:")
    print(f"  R² Scores: {scores}")
    print(f"  Mean R²: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Summary
baseline_df = pd.DataFrame(baseline_results)
print("\n" + "="*60)
print("BASELINE MODELS SUMMARY:")
print("="*60)
print(baseline_df.to_string(index=False))

print("\n✓ Baseline models established for comparison")

## Section 9: Model Candidate Pipelines

In [ ]:
print("="*60)
print("MODEL CANDIDATE PIPELINES")
print("="*60)

# Dictionary to store models and their hyperparameter grids
models = {}

# 1. Random Forest Regressor
print("\n1. RANDOM FOREST REGRESSOR:")
rf_params = {
    'n_estimators': randint(50, 300),
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'random_state': [42]
}
models['Random Forest'] = {
    'model': RandomForestRegressor(n_jobs=-1),
    'params': rf_params
}
print("   ✓ Configured with hyperparameter grid")

# 2. Gradient Boosting Regressor
print("\n2. GRADIENT BOOSTING REGRESSOR:")
gb_params = {
    'n_estimators': randint(100, 300),
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.6, 0.8, 1.0],
    'max_features': ['sqrt', 'log2'],
    'random_state': [42]
}
models['Gradient Boosting'] = {
    'model': GradientBoostingRegressor(),
    'params': gb_params
}
print("   ✓ Configured with hyperparameter grid")

# 3. XGBoost Regressor
print("\n3. XGBOOST REGRESSOR:")
xgb_params = {
    'n_estimators': randint(100, 300),
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'random_state': [42]
}
models['XGBoost'] = {
    'model': xgb.XGBRegressor(random_state=42),
    'params': xgb_params
}
print("   ✓ Configured with hyperparameter grid")

# 4. LightGBM Regressor
print("\n4. LIGHTGBM REGRESSOR:")
lgb_params = {
    'n_estimators': randint(100, 300),
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9, -1],
    'num_leaves': [31, 50, 100, 200],
    'subsample': [0.6, 0.8, 1.0],
    'random_state': [42]
}
models['LightGBM'] = {
    'model': lgb.LGBMRegressor(verbose=-1),
    'params': lgb_params
}
print("   ✓ Configured with hyperparameter grid")

# 5. Support Vector Regressor
print("\n5. SUPPORT VECTOR REGRESSOR (SVR):")
svr_params = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.1, 1, 10, 100],
    'epsilon': [0.01, 0.1, 0.2],
    'gamma': ['scale', 'auto']
}
models['SVR'] = {
    'model': SVR(),
    'params': svr_params
}
print("   ✓ Configured with hyperparameter grid")

# 6. K-Nearest Neighbors
print("\n6. K-NEAREST NEIGHBORS REGRESSOR:")
knn_params = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
models['KNN'] = {
    'model': KNeighborsRegressor(n_jobs=-1),
    'params': knn_params
}
print("   ✓ Configured with hyperparameter grid")

print(f"\n✓ {len(models)} model candidates configured")
print("\nModels ready for optimization:")
for i, name in enumerate(models.keys(), 1):
    print(f"   {i}. {name}")

## Section 10: Hyperparameter Optimization

In [ ]:
print("="*60)
print("HYPERPARAMETER OPTIMIZATION")
print("="*60)

# Store optimized models
optimized_models = {}

# Quick hyperparameter optimization using RandomizedSearchCV
print("\nRunning RandomizedSearchCV for each model...")
print("(This may take a few minutes)\n")

from sklearn.model_selection import RandomizedSearchCV

for model_name in ['Random Forest', 'Gradient Boosting', 'XGBoost', 'LightGBM']:
    print(f"Optimizing {model_name}...", end=" ")
    
    model = models[model_name]['model']
    params = models[model_name]['params']
    
    # Use RandomizedSearchCV for faster hyperparameter tuning
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=20,  # Number of parameter settings sampled
        cv=3,  # 3-fold CV for faster iteration
        scoring='r2',
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    
    # Fit on training data
    random_search.fit(X_train, y_train)
    
    optimized_models[model_name] = {
        'model': random_search.best_estimator_,
        'best_params': random_search.best_params_,
        'best_score': random_search.best_score_
    }
    
    print(f"Best CV R²: {random_search.best_score_:.4f}")
    print(f"  Best params: {random_search.best_params_}")

# Also optimize SVR and KNN (simpler models, faster optimization)
print(f"\nOptimizing SVR...", end=" ")
svr_model = models['SVR']['model']
svr_search = RandomizedSearchCV(
    svr_model, models['SVR']['params'], n_iter=15, cv=3, scoring='r2', 
    n_jobs=-1, random_state=42, verbose=0
)
svr_search.fit(X_train, y_train)
optimized_models['SVR'] = {
    'model': svr_search.best_estimator_,
    'best_params': svr_search.best_params_,
    'best_score': svr_search.best_score_
}
print(f"Best CV R²: {svr_search.best_score_:.4f}")

print(f"\nOptimizing KNN...", end=" ")
knn_model = models['KNN']['model']
knn_search = RandomizedSearchCV(
    knn_model, models['KNN']['params'], n_iter=15, cv=3, scoring='r2', 
    n_jobs=-1, random_state=42, verbose=0
)
knn_search.fit(X_train, y_train)
optimized_models['KNN'] = {
    'model': knn_search.best_estimator_,
    'best_params': knn_search.best_params_,
    'best_score': knn_search.best_score_
}
print(f"Best CV R²: {knn_search.best_score_:.4f}")

print("\n" + "="*60)
print("OPTIMIZATION SUMMARY")
print("="*60)

# Create summary dataframe
opt_summary = []
for model_name, opt_dict in optimized_models.items():
    opt_summary.append({
        'Model': model_name,
        'Best CV R²': opt_dict['best_score'],
        'Train Samples': X_train.shape[0],
        'Features Used': X_train.shape[1]
    })

opt_summary_df = pd.DataFrame(opt_summary).sort_values('Best CV R²', ascending=False)
print(opt_summary_df.to_string(index=False))

print("\n✓ Hyperparameter optimization completed!")

## Section 11: Cross-Validated Training & Model Comparison

In [ ]:
print("="*60)
print("CROSS-VALIDATED TRAINING & MODEL COMPARISON")
print("="*60)

# Cross-validation on optimized models
cv_results = {}
all_predictions = {}

print("\nRunning 5-Fold Cross-Validation on optimized models...\n")

scoring_metrics = ['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']

for model_name, opt_dict in optimized_models.items():
    print(f"Evaluating {model_name}...", end=" ")
    
    model = opt_dict['model']
    
    # Cross-validate with multiple metrics
    cv_results[model_name] = cross_validate(
        model, X_train, y_train, cv=cv, scoring=scoring_metrics, n_jobs=-1
    )
    
    # Get predictions on val set for visualization
    val_pred = model.predict(X_val)
    all_predictions[model_name] = {
        'val_pred': val_pred,
        'val_true': y_val.values
    }
    
    # Print CV results
    r2_scores = cv_results[model_name]['test_r2']
    print(f"R² Mean: {r2_scores.mean():.4f} (+/- {r2_scores.std():.4f})")

# Comprehensive comparison
print("\n" + "="*60)
print("DETAILED CROSS-VALIDATION RESULTS")
print("="*60)

comparison_data = []
for model_name, cv_dict in cv_results.items():
    r2_scores = cv_dict['test_r2']
    mse_scores = -cv_dict['test_neg_mean_squared_error']
    mae_scores = -cv_dict['test_neg_mean_absolute_error']
    rmse_scores = np.sqrt(mse_scores)
    
    comparison_data.append({
        'Model': model_name,
        'CV R² Mean': r2_scores.mean(),
        'CV R² Std': r2_scores.std(),
        'CV RMSE Mean': rmse_scores.mean(),
        'CV RMSE Std': rmse_scores.std(),
        'CV MAE Mean': mae_scores.mean(),
        'CV MAE Std': mae_scores.std()
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('CV R² Mean', ascending=False)
print(comparison_df.to_string(index=False))

# Visualization: Compare models
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# R² Scores
r2_data = [cv_results[name]['test_r2'] for name in comparison_df['Model']]
axes[0].boxplot(r2_data, labels=comparison_df['Model'])
axes[0].set_ylabel('R² Score')
axes[0].set_title('Cross-Validation R² Distribution')
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# RMSE Scores
rmse_data = [np.sqrt(-cv_results[name]['test_neg_mean_squared_error']) 
             for name in comparison_df['Model']]
axes[1].boxplot(rmse_data, labels=comparison_df['Model'])
axes[1].set_ylabel('RMSE ($)')
axes[1].set_title('Cross-Validation RMSE Distribution')
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

# MAE Scores
mae_data = [-cv_results[name]['test_neg_mean_absolute_error'] 
            for name in comparison_df['Model']]
axes[2].boxplot(mae_data, labels=comparison_df['Model'])
axes[2].set_ylabel('MAE ($)')
axes[2].set_title('Cross-Validation MAE Distribution')
axes[2].grid(axis='y', alpha=0.3)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Best model selection
best_model_name = comparison_df.iloc[0]['Model']
best_model = optimized_models[best_model_name]['model']

print(f"\n{'='*60}")
print(f"BEST MODEL SELECTED: {best_model_name}")
print(f"{'='*60}")
print(f"CV R² Mean: {comparison_df.iloc[0]['CV R² Mean']:.4f}")
print(f"CV RMSE Mean: ${comparison_df.iloc[0]['CV RMSE Mean']:.4f}")
print(f"CV MAE Mean: ${comparison_df.iloc[0]['CV MAE Mean']:.4f}")

## Section 12: Model Evaluation & Metrics

In [ ]:
print("="*60)
print("FINAL MODEL EVALUATION ON VALIDATION & TEST SETS")
print("="*60)

# Make predictions with best model
best_model.fit(X_train, y_train)  # Refit on full training set

y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)
y_test_pred = best_model.predict(X_test)

# Calculate metrics
def calc_metrics(y_true, y_pred, set_name):
    """Calculate all regression metrics"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # MAPE (Mean Absolute Percentage Error)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    return {
        'Set': set_name,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'R²': r2,
        'Samples': len(y_true)
    }

metrics_train = calc_metrics(y_train, y_train_pred, 'Train')
metrics_val = calc_metrics(y_val, y_val_pred, 'Validation')
metrics_test = calc_metrics(y_test, y_test_pred, 'Test')

print(f"\n{best_model_name} - PERFORMANCE METRICS:")
print("="*60)
metrics_df = pd.DataFrame([metrics_train, metrics_val, metrics_test])
print(metrics_df.to_string(index=False))

# Detailed diagnostics
print(f"\n{'='*60}")
print("DETAILED REGRESSION METRICS EXPLANATION")
print(f"{'='*60}")
print(f"""
RMSE (Root Mean Squared Error): ${metrics_test['RMSE']:.4f}
  - Typical prediction error magnitude
  - Formula: √(Σ(y_true - y_pred)² / n)
  
MAE (Mean Absolute Error): ${metrics_test['MAE']:.4f}
  - Average absolute error in same units as target
  - More robust to outliers than RMSE
  
MAPE (Mean Absolute Percentage Error): {metrics_test['MAPE']:.2f}%
  - Percentage error relative to actual values
  - Good for comparing across different scales
  
R² Score: {metrics_test['R²']:.4f}
  - Proportion of variance explained (0-1)
  - Formula: 1 - (SS_res / SS_tot)
  - Higher is better; 1.0 = perfect prediction
""")

# Prediction Analysis
print(f"\n{'='*60}")
print("PREDICTION ANALYSIS")
print(f"{'='*60}")

# Prediction residuals
train_residuals = y_train - y_train_pred
val_residuals = y_val.values - y_val_pred
test_residuals = y_test.values - y_test_pred

print(f"\nResiduals Statistics (Test Set):")
print(f"  Mean:  ${test_residuals.mean():.4f} (ideally close to 0)")
print(f"  Std:   ${test_residuals.std():.4f}")
print(f"  Min:   ${test_residuals.min():.4f}")
print(f"  Max:   ${test_residuals.max():.4f}")

# Visualization: Predictions vs Actual
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Actual vs Predicted (Validation)
axes[0, 0].scatter(y_val.values, y_val_pred, alpha=0.6, s=50)
min_val = min(y_val.min(), y_val_pred.min())
max_val = max(y_val.max(), y_val_pred.max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[0, 0].set_xlabel('Actual Cost ($)')
axes[0, 0].set_ylabel('Predicted Cost ($)')
axes[0, 0].set_title(f'Validation: Actual vs Predicted (R²={metrics_val["R²"]:.4f})')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Actual vs Predicted (Test)
axes[0, 1].scatter(y_test.values, y_test_pred, alpha=0.6, s=50, color='orange')
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[0, 1].set_xlabel('Actual Cost ($)')
axes[0, 1].set_ylabel('Predicted Cost ($)')
axes[0, 1].set_title(f'Test: Actual vs Predicted (R²={metrics_test["R²"]:.4f})')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Residuals distribution (Test)
axes[1, 0].hist(test_residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(0, color='r', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Residuals ($)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Residuals (Test Set)')
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Residuals vs Predicted (Test)
axes[1, 1].scatter(y_test_pred, test_residuals, alpha=0.6, s=50, color='green')
axes[1, 1].axhline(0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Predicted Cost ($)')
axes[1, 1].set_ylabel('Residuals ($)')
axes[1, 1].set_title('Residuals vs Predictions (Test Set)')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Model evaluation completed!")

## Section 13: Model Interpretation with SHAP

In [ ]:
print("="*60)
print("MODEL INTERPRETATION WITH SHAP")
print("="*60)

# SHAP interpretation (works well for tree-based models)
if best_model_name in ['Random Forest', 'Gradient Boosting', 'XGBoost', 'LightGBM']:
    print(f"\nGenerating SHAP explanations for {best_model_name}...")
    print("(This may take a moment for large datasets)\n")
    
    try:
        # Create SHAP explainer
        explainer = shap.TreeExplainer(best_model)
        
        # Calculate SHAP values for a sample (using train set for speed)
        sample_size = min(100, len(X_train))
        shap_values = explainer.shap_values(X_train.iloc[:sample_size])
        
        # Summary plot - global feature importance
        print("Generating SHAP summary plot...")
        plt.figure(figsize=(12, 6))
        shap.summary_plot(shap_values, X_train.iloc[:sample_size], 
                         feature_names=final_features, show=False)
        plt.title(f'{best_model_name} - Feature Importance (SHAP Summary Plot)')
        plt.tight_layout()
        plt.show()
        
        # Bar plot for average impact
        print("Generating SHAP bar plot...")
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_train.iloc[:sample_size], 
                         feature_names=final_features, plot_type="bar", show=False)
        plt.title(f'{best_model_name} - Average Feature Importance (SHAP)')
        plt.tight_layout()
        plt.show()
        
        print("\n✓ SHAP interpretation completed!")
        
    except Exception as e:
        print(f"Note: SHAP visualization encountered an issue: {str(e)}")
        print("Continuing with model feature importance instead...")
        
        # Fallback to model feature importance
        if hasattr(best_model, 'feature_importances_'):
            importances = best_model.feature_importances_
            importance_df = pd.DataFrame({
                'Feature': final_features,
                'Importance': importances
            }).sort_values('Importance', ascending=False)
            
            plt.figure(figsize=(10, 6))
            plt.barh(importance_df['Feature'], importance_df['Importance'])
            plt.xlabel('Importance')
            plt.title(f'{best_model_name} - Feature Importance')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.show()
            
            print("\nTop 10 Features by Importance:")
            print(importance_df.head(10).to_string(index=False))

else:
    print(f"\nNote: {best_model_name} is not a tree-based model.")
    print("Using Permutation Importance instead...\n")
    
    from sklearn.inspection import permutation_importance
    
    perm_importance = permutation_importance(best_model, X_val, y_val, n_repeats=10, 
                                            random_state=42, n_jobs=-1)
    
    perm_imp_df = pd.DataFrame({
        'Feature': final_features,
        'Importance': perm_importance.importances_mean
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    plt.barh(perm_imp_df['Feature'].head(10), perm_imp_df['Importance'].head(10))
    plt.xlabel('Permutation Importance')
    plt.title(f'{best_model_name} - Top 10 Features (Permutation Importance)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("Top 10 Features by Permutation Importance:")
    print(perm_imp_df.head(10).to_string(index=False))

## Section 14: Final Test, Persistence & Reproducibility

In [ ]:
print("="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)

print(f"""
╔══════════════════════════════════════════════════════════════╗
║          ML PIPELINE EXECUTION SUMMARY                      ║
╚══════════════════════════════════════════════════════════════╝

1. DATASET ANALYSIS
   - Total instances: {len(df)}
   - Features analyzed: {len(numeric_cols)}
   - Features selected: {len(final_features)}
   - Missing values handled: ✓
   - Outliers clipped: ✓

2. TRAIN/VAL/TEST SPLIT (60/20/20)
   - Train samples: {X_train.shape[0]}
   - Validation samples: {X_val.shape[0]}
   - Test samples: {X_test.shape[0]}

3. MODELS EVALUATED
   - Baseline models: 3
   - Advanced models: 6
   - Total configurations: 9+

4. BEST MODEL: {best_model_name}
   - CV R² Mean: {metrics_val['R²']:.4f}
   - Test RMSE: ${metrics_test['RMSE']:.4f}
   - Test MAE: ${metrics_test['MAE']:.4f}
   - Test R²: {metrics_test['R²']:.4f}

5. FEATURE IMPORTANCE
   - Top feature: {final_features[0] if len(final_features) > 0 else 'N/A'}
   - Total features used: {len(final_features)}

6. MODEL ARTIFACTS
   ✓ Model parameters optimized
   ✓ Cross-validation performed
   ✓ SHAP interpretations generated
   ✓ Predictions evaluated

╔══════════════════════════════════════════════════════════════╗
║              ✓ PIPELINE COMPLETED SUCCESSFULLY              ║
╚══════════════════════════════════════════════════════════════╝
""")

# Save model and preprocessing objects
print("\nSaving model artifacts...")
output_dir = Path('c:/Users/15132000230/Documents/GitHub/sintance-DataScientistSPV-KOMDIGI/models')
output_dir.mkdir(exist_ok=True)

# Save model
model_path = output_dir / f'{best_model_name.replace(" ", "_")}_model.pkl'
joblib.dump(best_model, model_path)
print(f"✓ Model saved: {model_path}")

# Save preprocessor
preprocessor_path = output_dir / 'preprocessor.pkl'
joblib.dump(preprocessor, preprocessor_path)
print(f"✓ Preprocessor saved: {preprocessor_path}")

# Save feature list
features_path = output_dir / 'selected_features.pkl'
joblib.dump(final_features, features_path)
print(f"✓ Features saved: {features_path}")

# Save metrics
metrics_path = output_dir / 'metrics.pkl'
metrics_all = {'train': metrics_train, 'val': metrics_val, 'test': metrics_test}
joblib.dump(metrics_all, metrics_path)
print(f"✓ Metrics saved: {metrics_path}")

# Create reproducibility report
print("\nGenerating reproducibility report...")

report = f"""
# EC2 INSTANCE PRICING PREDICTION - MODEL REPORT

## Reproducibility Information
- **Execution Date**: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
- **Random State**: {random_state}
- **Python Version**: 3.8+
- **Key Libraries**: scikit-learn, xgboost, lightgbm, shap, pandas, numpy

## Dataset Information
- **Source**: instance_comparison.csv
- **Total Records**: {len(df)}
- **Target Variable**: Linux On Demand Cost
- **Feature Count**: {len(final_features)} (from {len(numeric_cols)} candidates)

## Data Split
- **Train Set**: {X_train.shape[0]} samples (60%)
- **Validation Set**: {X_val.shape[0]} samples (20%)
- **Test Set**: {X_test.shape[0]} samples (20%)

## Best Model: {best_model_name}

### Hyperparameters
{optimized_models[best_model_name]['best_params']}

### Performance Metrics (Test Set)
- **RMSE**: ${metrics_test['RMSE']:.4f}
- **MAE**: ${metrics_test['MAE']:.4f}
- **MAPE**: {metrics_test['MAPE']:.2f}%
- **R² Score**: {metrics_test['R²']:.4f}

### Selected Features ({len(final_features)})
{', '.join(final_features)}

## Model Artifacts Location
- Model: {model_path}
- Preprocessor: {preprocessor_path}
- Features: {features_path}
- Metrics: {metrics_path}

## How to Use Saved Model
```python
import joblib
import pandas as pd

# Load artifacts
model = joblib.load('models/{best_model_name.replace(" ", "_")}_model.pkl')
preprocessor = joblib.load('models/preprocessor.pkl')
features = joblib.load('models/selected_features.pkl')

# Preprocess new data
X_new_preprocessed = preprocessor.transform(X_new[features])

# Make predictions
predictions = model.predict(X_new_preprocessed)
```

## Cross-Validation Results
All models were evaluated using 5-Fold Cross-Validation.
See detailed results in the notebook sections.

---
Generated by EC2 Pricing Prediction Pipeline
"""

report_path = output_dir / 'MODEL_REPORT.md'
with open(report_path, 'w') as f:
    f.write(report)
print(f"✓ Report saved: {report_path}")

print("\n" + "="*60)
print("✓ ALL ARTIFACTS SAVED SUCCESSFULLY!")
print("="*60)

# Final test prediction example
print("\n" + "="*60)
print("EXAMPLE PREDICTION")
print("="*60)

# Use first sample from test set
example_idx = 0
example_features = X_test.iloc[[example_idx]]
example_actual = y_test.values[example_idx]
example_pred = best_model.predict(example_features)[0]
example_error = abs(example_actual - example_pred)
example_error_pct = (example_error / example_actual) * 100

print(f"\nSample Instance:")
print(f"  Actual Cost: ${example_actual:.4f}/hour")
print(f"  Predicted Cost: ${example_pred:.4f}/hour")
print(f"  Error: ${example_error:.4f} ({example_error_pct:.2f}%)")

print("\n✓ COMPLETE MODELING PIPELINE FINISHED!")